# Extended R > -1/2 Computation
## Verify corrRatio(p) for all primes with M(p) <= -3, up to p = 1000

**Goal:** Compute R = crossTerm(p) / (2 * shiftSquaredSum(p)) for all primes p
where M(p) <= -3, and verify R > -1/2 for each. Uses exact rational arithmetic.

**Key formula:**
- B + C = delta_sq * (1 + R)
- Need R > -1 for B+C > 0 (our claim: R > -1/2)
- Empirically: |R| < 0.52 for all primes p >= 11

**Runtime:** ~2 min (Quick to p=200), ~15 min (Full to p=1000)

**GPU:** Not required.

In [ ]:
# --- CELL 1: Configuration ---
MODE = 'quick'  # 'quick' -> p<=200, 'full' -> p<=1000

QUICK_LIMIT = 200
FULL_LIMIT = 1000
LIMIT = QUICK_LIMIT if MODE == 'quick' else FULL_LIMIT

import time
from math import gcd, isqrt, sqrt
from fractions import Fraction
from collections import defaultdict
import json

print(f'Mode: {MODE}, checking primes up to {LIMIT}')

In [ ]:
# --- CELL 2: Sieve and Mertens function ---

def sieve_primes(limit):
    sieve = [True] * (limit + 1)
    sieve[0] = sieve[1] = False
    for i in range(2, isqrt(limit) + 1):
        if sieve[i]:
            for j in range(i*i, limit + 1, i):
                sieve[j] = False
    return [i for i in range(2, limit + 1) if sieve[i]]

def compute_mobius_mertens(limit):
    """Compute Mobius mu(n) and Mertens M(n) for n=1..limit."""
    mu = [0] * (limit + 1)
    mu[1] = 1
    smallest_prime = [0] * (limit + 1)
    for i in range(2, limit + 1):
        if smallest_prime[i] == 0:
            for j in range(i, limit + 1, i):
                if smallest_prime[j] == 0:
                    smallest_prime[j] = i
    for n in range(2, limit + 1):
        p = smallest_prime[n]
        if (n // p) % p == 0:
            mu[n] = 0
        else:
            mu[n] = -mu[n // p]
    M = [0] * (limit + 1)
    s = 0
    for i in range(1, limit + 1):
        s += mu[i]
        M[i] = s
    return mu, M

primes = sieve_primes(LIMIT)
mu, M = compute_mobius_mertens(LIMIT)

# Filter: only primes with M(p) <= -3
target_primes = [p for p in primes if p >= 11 and M[p] <= -3]
print(f'Primes up to {LIMIT}: {len(primes)}')
print(f'Primes with M(p) <= -3: {len(target_primes)}')
if target_primes:
    print(f'First 10: {target_primes[:10]}')
    print(f'Their M(p): {[M[p] for p in target_primes[:10]]}')

In [ ]:
# --- CELL 3: Exact R computation using Fraction arithmetic ---

def build_farey(N):
    """Build Farey sequence F_N as list of (numerator, denominator) tuples."""
    fracs = []
    a, b = 0, 1
    c, d = 1, N
    fracs.append((0, 1))
    fracs.append((1, N))
    while (c, d) != (1, 1):
        k = (N + b) // d
        a, b, c, d = c, d, k*c - a, k*d - b
        fracs.append((c, d))
    return fracs

def compute_R_exact(p):
    """Compute R = crossTerm / (2 * shiftSquaredSum) using exact Fraction arithmetic."""
    N = p - 1
    fracs = build_farey(N)
    n = len(fracs)

    # For each fraction a/b in F_N:
    #   D(a/b) = rank - n * (a/b)    [rank discrepancy]
    #   delta(a/b) = (a - sigma_p(a,b)) / b  where sigma_p(a) = (p*a) mod b
    crossTerm = Fraction(0)
    shiftSquaredSum = Fraction(0)

    for j, (a_j, b_j) in enumerate(fracs):
        # Rank discrepancy D = j - n*(a_j/b_j)
        D = Fraction(j) - Fraction(n * a_j, b_j)

        # Shift delta = (a_j - (p*a_j mod b_j)) / b_j
        sigma = (p * a_j) % b_j
        delta = Fraction(a_j - sigma, b_j)

        crossTerm += D * delta
        shiftSquaredSum += delta * delta

    crossTerm *= 2  # B_raw = 2 * sum(D * delta)

    if shiftSquaredSum == 0:
        return None

    R = crossTerm / (2 * shiftSquaredSum)  # corrRatio
    B_plus_C = crossTerm + shiftSquaredSum
    one_plus_R_check = B_plus_C / shiftSquaredSum

    return {
        'p': p,
        'M_p': M[p],
        'n_farey': n,
        'crossTerm': crossTerm,
        'shiftSquaredSum': shiftSquaredSum,
        'R': R,
        'R_float': float(R),
        'one_plus_R': float(one_plus_R_check),
        'B_plus_C_positive': B_plus_C > 0,
    }

# Quick sanity check
test = compute_R_exact(11)
print(f'Sanity check p=11: R={test["R_float"]:.6f}, 1+R={test["one_plus_R"]:.6f}, B+C>0: {test["B_plus_C_positive"]}')

In [ ]:
# --- CELL 4: Run computation for all target primes ---
t0 = time.time()

all_results = []
violations = []  # R <= -1/2
min_R = float('inf')
min_R_p = 0
max_abs_R = 0
max_abs_R_p = 0

print(f'{"p":>6} {"M(p)":>5} {"R":>12} {"1+R":>10} {"B+C>0":>7} {"Time":>8}')
print('-' * 55)

for i, p in enumerate(target_primes):
    r = compute_R_exact(p)
    if r is None:
        continue

    all_results.append(r)
    R_f = r['R_float']

    if R_f < min_R:
        min_R = R_f
        min_R_p = p
    if abs(R_f) > max_abs_R:
        max_abs_R = abs(R_f)
        max_abs_R_p = p
    if R_f <= -0.5:
        violations.append(r)

    elapsed = time.time() - t0
    # Print selected rows
    if i < 10 or i % 20 == 0 or p == target_primes[-1] or R_f <= -0.4:
        print(f'{p:6d} {r["M_p"]:5d} {R_f:12.6f} {r["one_plus_R"]:10.6f} {str(r["B_plus_C_positive"]):>7} {elapsed:8.1f}s')

total_time = time.time() - t0
print(f'\nDone in {total_time:.1f}s')

In [ ]:
# --- CELL 5: Results summary ---

print('=' * 60)
print('R > -1/2 VERIFICATION RESULTS')
print('=' * 60)
print(f'Primes tested: {len(all_results)}')
print(f'All with M(p) <= -3, up to p = {LIMIT}')
print()
print(f'min(R)      = {min_R:.6f}  at p = {min_R_p}')
print(f'max(|R|)    = {max_abs_R:.6f}  at p = {max_abs_R_p}')
print(f'min(1+R)    = {min(r["one_plus_R"] for r in all_results):.6f}')
print()

all_positive = all(r['B_plus_C_positive'] for r in all_results)
all_above_half = len(violations) == 0

print(f'B+C > 0 for ALL tested primes: {all_positive}')
print(f'R > -1/2 for ALL tested primes: {all_above_half}')
print(f'Violations (R <= -1/2): {len(violations)}')

if violations:
    print('\nVIOLATIONS:')
    for v in violations:
        print(f'  p={v["p"]}, M(p)={v["M_p"]}, R={v["R_float"]:.6f}')
else:
    print('\n*** VERIFIED: R > -1/2 for all tested primes with M(p) <= -3 ***')

In [ ]:
# --- CELL 6: Also test ALL primes (not just M(p)<=-3) ---
print('\nNow testing ALL primes >= 11 (regardless of M(p))...')
t0 = time.time()

all_primes_test = [p for p in primes if p >= 11]
all_R_values = []
all_violations = []

for p in all_primes_test:
    r = compute_R_exact(p)
    if r:
        all_R_values.append((p, r['R_float'], r['M_p']))
        if r['R_float'] <= -0.5:
            all_violations.append(r)

elapsed = time.time() - t0
R_vals = [x[1] for x in all_R_values]
print(f'Tested {len(all_R_values)} primes in {elapsed:.1f}s')
print(f'min(R) = {min(R_vals):.6f}')
print(f'max(R) = {max(R_vals):.6f}')
print(f'R > -1/2 for ALL: {len(all_violations) == 0}')

In [ ]:
# --- CELL 7: Visualization ---
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ps = [x[0] for x in all_R_values]
Rs = [x[1] for x in all_R_values]
Ms = [x[2] for x in all_R_values]

# 1. R vs p
ax = axes[0]
ax.scatter(ps, Rs, s=5, alpha=0.5)
ax.axhline(y=-0.5, color='r', linestyle='--', label='R = -1/2')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Prime p')
ax.set_ylabel('R(p)')
ax.set_title('corrRatio R(p) vs prime p')
ax.legend()

# 2. R vs M(p)
ax = axes[1]
ax.scatter(Ms, Rs, s=5, alpha=0.5)
ax.axhline(y=-0.5, color='r', linestyle='--', label='R = -1/2')
ax.set_xlabel('M(p)')
ax.set_ylabel('R(p)')
ax.set_title('corrRatio vs Mertens function')
ax.legend()

# 3. Distribution of R
ax = axes[2]
ax.hist(Rs, bins=40, alpha=0.7)
ax.axvline(x=-0.5, color='r', linestyle='--', label='R = -1/2')
ax.set_xlabel('R')
ax.set_ylabel('Count')
ax.set_title('Distribution of R(p)')
ax.legend()

plt.tight_layout()
plt.savefig('r_bound_verification.png', dpi=150)
plt.show()
print('Plot saved to r_bound_verification.png')

In [ ]:
# --- CELL 8: Save results ---
output = {
    'mode': MODE,
    'limit': LIMIT,
    'primes_tested': len(all_R_values),
    'target_primes_M_leq_minus3': len(all_results),
    'min_R': min(R_vals),
    'max_R': max(R_vals),
    'all_above_minus_half': len(all_violations) == 0,
    'all_B_plus_C_positive': all_positive,
    'R_values': [(p, float(r), int(m)) for p, r, m in all_R_values],
}

with open('r_bound_verification_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Results saved to r_bound_verification_results.json')

# Google Drive save
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    import shutil
    dest = '/content/drive/MyDrive/Farey_Results/'
    import os; os.makedirs(dest, exist_ok=True)
    shutil.copy('r_bound_verification_results.json', dest)
    shutil.copy('r_bound_verification.png', dest)
    print(f'Copied to Google Drive: {dest}')
except:
    print('Google Drive not mounted. Results saved locally.')